# NHCS Scenario 5 — How long do patients stay, and what keeps them: length of stay, childbirth and procedures

_A numerical outcome and a linear model on half a million discharges. Inpatient dataset only. Level: second analysis. About 90 minutes._

**Before you start.** The NHCS pathology holds two datasets from about 200 US hospitals, 2020 to
2022: 1,368,648 emergency department visit records and 488,636 inpatient discharge records, each a
5% sample of all encounters of the participating hospitals. **A row is an encounter record, not a
patient**, and the two datasets cannot be linked: each is an independent sample, the identifiers
were renumbered, and NCHS merged the ED outcome "admitted" into "other". The MIP does not use the
survey weights: every figure describes *the sampled encounters*, not US hospital care. Codes are
NCHS codes: Sex 1 = Male, 2 = Female; NCHS flags 1 = Yes, 2 = No; curated flags 0 = No, 1 = Yes;
age is top-coded at 86 and the length of stay at 15 days.

**How this notebook works.** Every analysis is a *federated* call: the code selects a dataset and
variables, defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a
model). No record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect. These are the largest datasets of this MIP: the notebook prints how
long the regressions take.

**Learning goals.** Describe a skewed, top-coded duration with medians and a histogram; compare it
across diagnosis groups and comorbidity levels (one-way ANOVA); describe the commonest reason for
a hospital stay, childbirth, and its caesarean rate; relate the stay to procedures and to the
new-technology drugs of the pandemic; and fit a linear regression that says how many days each
factor adds.

**Clinical background.** The length of stay is the hospital's central operational measure: it
decides how many beds a population needs. It is driven by the reason for admission (a delivery
lasts two days, sepsis a week), by comorbidity, by what is done during the stay, and by what the
patient needs at discharge. NHCS records the stay in days, top-coded at 15 (every longer stay reads
15, with a separate flag for 30 days or more), up to 10 procedures, and flags for deliveries,
caesarean sections, transfusions, imaging and "new technology" procedures, which in these years
are mostly the COVID-19 drugs such as remdesivir. The top code flattens the long stays: means are
underestimated, medians are exact.

## 1. Connect to the MIP and open the NHCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHCS")        # add version="ED_Inpatient_v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the platform labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

ed = dataset("nhcs_ed", "NHCS emergency department visits, USA, 2020-2022")
ip = dataset("nhcs_ip", "NHCS inpatient discharges, USA, 2020-2022")
display(dm)
display(dm.datasets.to_frame())

In [ ]:
# The variable tree is the anatomy of one encounter record: Encounter, Patient, Discharge,
# Diagnoses (with the curated condition flags and the NCHS CCSR flags), and the branch
# Inpatient stay (empty for ED visits): length of stay and procedures.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one Dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_summary(result, variables, dataset=POOLED):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v, dataset):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable, dataset=POOLED):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable, dataset):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan, "missing": (r.get("data") or {}).get("num_na")})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",), dataset=POOLED):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable, dataset)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None, datasets=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def band_means(numeric, bands, base_filter=None, datasets=None):
    """n, mean and median of a numerical variable in bands given as {label: filter}."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({"band": label, "n": d.get("num_dtps"), "mean": d.get("mean"), "median": d.get("q2")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None, datasets=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f, datasets=datasets).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True, datasets=datasets).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1", datasets=None):
    """% of records with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f, datasets=datasets).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True, datasets=datasets).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def month_label(i):
    """Month index 1..36 -> '2020-01' .. '2022-12'."""
    return f"{2020 + (i - 1) // 12}-{(i - 1) % 12 + 1:02d}"


def by_month(outcomes, variable, months=range(1, 37), datasets=None):
    """% of records with each outcome (code 1) per month of the time axis: one describe per month."""
    rows = []
    for m in months:
        res = analysis(outcomes, F(variable) == m, datasets=datasets).describe()
        row = {"month": m, "label": month_label(m)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get("1", 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None, rotate=0):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:,.0f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(7, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def n_fmt(value):
    # a count with thousands separators, or n/a when the platform did not return it
    return f"{value:,}" if isinstance(value, (int, float)) else "n/a"


def ttest_from_describe(result_a, result_b, variable):
    """The pooled-variance t statistic of two groups from their describe results (n, mean, SD):
    all a t-test needs, for groups the platform's t-test cannot form (several codes pooled)."""
    a = numeric_summary(result_a, [variable]).iloc[0]
    b = numeric_summary(result_b, [variable]).iloc[0]
    n1, n2 = a["n"], b["n"]
    sp2 = ((n1 - 1) * a["sd"] ** 2 + (n2 - 1) * b["sd"] ** 2) / (n1 + n2 - 2)
    t = (a["mean"] - b["mean"]) / math.sqrt(sp2 * (1 / n1 + 1 / n2))
    return {"mean_a": a["mean"], "mean_b": b["mean"], "diff": a["mean"] - b["mean"], "t": t, "n": n1 + n2}


def term_label(term, variables):
    """Model term -> readable label: 'SEX[2]' -> 'Sex = Female', 'AGE' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def linear_table(result, variables):
    """Linear regression result -> one row per term with coefficient (units of the outcome), 95% CI and p."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return pd.DataFrame({"term": [term_label(t, variables) for t in raw.get("indep_vars", [])],
                         "coefficient": raw.get("coefficients"), "ci_low": raw.get("lower_ci"),
                         "ci_high": raw.get("upper_ci"), "p": raw.get("pvalues")})


def show_coefficients(table, title, unit="days"):
    """Table and dot chart of linear-regression coefficients (the intercept left out of the chart)."""
    display(table.style.format({"coefficient": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table[table["term"] != "Intercept"].iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["coefficient"], range(len(t)),
                xerr=[t["coefficient"] - t["ci_low"], t["ci_high"] - t["coefficient"]], fmt="o", capsize=4)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel(f"{unit} added, for the same values of the other predictors")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The dataset and the variables of this scenario

| Code | Label in the MIP | Tree location (under NHCS) | Role | Values |
|---|---|---|---|---|
| `LOS` | Length of stay | Inpatient stay (empty for ED visits) / Length of stay | outcome: days, top-coded at 15 | integer, 0-15 days |
| `los_group` | Length of stay group | Inpatient stay (empty for ED visits) / Length of stay | the stay in bands; curation-derived | 0 Same day (0 days); 1 1-2 days; 2 3-4 days; 3 5-7 days; 4 8-14 days; 5 15 days or more |
| `LOS_30DAYS` | Stay of 30 days or more | Inpatient stay (empty for ED visits) / Length of stay | stays of 30 days or more | 1 Length of stay 30 days or longer; 2 Length of stay less than 30 days |
| `dx1_group` | First-listed diagnosis, group | Diagnoses | reason for admission; curation-derived | 40 codes, listed in the tree |
| `charlson_group` | Charlson index group | Diagnoses / Condition flags (curated) | comorbidity; ANOVA; curation-derived | 0 0 (no Charlson condition); 1 1-2; 2 3-4; 3 5 or more |
| `cond_delivery` | Delivery (mother's record) | Diagnoses / Condition flags (curated) | cohort filter of Step 4; curation-derived | 0 No; 1 Yes |
| `pr_cesarean` | Caesarean section | Inpatient stay (empty for ED visits) / Procedures | caesarean section; curation-derived | 0 No; 1 Yes |
| `pr_vaginal_delivery` | Vaginal delivery | Inpatient stay (empty for ED visits) / Procedures | vaginal delivery; curation-derived | 0 No; 1 Yes |
| `age_group` | Age group | Patient | caesarean by age; extension; curation-derived | 1 Under 1 year; 2 1-17 years; 3 18-44 years; 4 45-64 years; 5 65-74 years; 6 75-84 years; 7 85 years and over |
| `YEAR` | Survey year | Encounter | caesarean and new technology by year | 2020 2020; 2021 2021; 2022 2022 |
| `pr_any` | Any procedure | Inpatient stay (empty for ED visits) / Procedures | any procedure; curation-derived | 0 No; 1 Yes |
| `n_pr` | Number of procedures | Inpatient stay (empty for ED visits) / Procedures | number of procedures; predictor; curation-derived | integer, 0-10 |
| `pr_transfusion` | Transfusion | Inpatient stay (empty for ED visits) / Procedures | procedure flag; curation-derived | 0 No; 1 Yes |
| `pr_imaging` | Imaging procedure | Inpatient stay (empty for ED visits) / Procedures | procedure flag; curation-derived | 0 No; 1 Yes |
| `pr_new_technology` | New technology procedure | Inpatient stay (empty for ED visits) / Procedures | procedure flag: mostly COVID-19 drugs; curation-derived | 0 No; 1 Yes |
| `covid_dx` | COVID-19, any position | Diagnoses / Condition flags (curated) | predictor; curation-derived | 0 No; 1 Yes |
| `NEWBORN` | Newborn | Patient | cohort filter of the model | 1 Newborn; 2 Not a newborn |
| `AGE` | Age | Patient | predictor | integer, 0-86 years |
| `SEX` | Sex | Patient | predictor | 1 Male; 2 Female |
| `sepsis_dx` | Sepsis, any position | Diagnoses / Condition flags (curated) | predictor; curation-derived | 0 No; 1 Yes |
| `discharge_group` | Discharge group | Discharge | extension; curation-derived | 1 Home, routine; 2 Home with home health care; 3 Transfer to another facility or hospice; 4 Left against medical advice; 5 Died; 6 Other (for ED visits includes admission) |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `AGE`) is what the csv
column is called; you will see it in some result payloads. The first line below chooses the
dataset(s) every call of this notebook uses.

In [ ]:
DATASETS = [ip]                          # every call of this notebook runs on the inpatient discharges

los        = dm.variables["Length of stay"]
los_group  = dm.variables["Length of stay group"]
los30      = dm.variables["Stay of 30 days or more"]
group      = dm.variables["First-listed diagnosis, group"]
charlson   = dm.variables["Charlson index group"]
delivery   = dm.variables["Delivery (mother's record)"]
cesarean   = dm.variables["Caesarean section"]
vaginal    = dm.variables["Vaginal delivery"]
age_group  = dm.variables["Age group"]
year       = dm.variables["Survey year"]
pr_any     = dm.variables["Any procedure"]
n_pr       = dm.variables["Number of procedures"]
transfusion = dm.variables["Transfusion"]
imaging    = dm.variables["Imaging procedure"]
newtech    = dm.variables["New technology procedure"]
covid      = dm.variables["COVID-19, any position"]
newborn    = dm.variables["Newborn"]
age        = dm.variables["Age"]
sex        = dm.variables["Sex"]
sepsis     = dm.variables["Sepsis, any position"]
outcome    = dm.variables["Discharge group"]
display(mip.to_frame([los, los_group, los30, group, charlson, delivery, cesarean, vaginal, age_group, year, pr_any, n_pr,
                      transfusion, imaging, newtech, covid, newborn, age, sex, sepsis, outcome]))

## Step 1. The distribution

*Expected:* 488,616 stays with a length: mean 4.66 days (SD 4.11), median 3, quartiles 2 to 6;
7.2% at the top code of 15 days or more, 1.9% (9,057 stays) of 30 days or more. Groups: same day
2.6%, 1 to 2 days 37.8%, 3 to 4 days 24.7%, 5 to 7 days 15.8%, 8 to 14 days 12.0%, 15 or more
7.2%. The histogram (16 bins, one per day) peaks at 2 days (107,064 stays) and falls steadily to a
last bar that holds the 35,145 top-coded stays. A typical stay is short; the mean is pulled up by
a long tail that the top code cuts off. Report the median, and say "15 or more" rather than 15.

In [ ]:
res = analysis([los, los_group, los30]).describe()
display(numeric_summary(res, [los]).style.format({c: "{:.2f}" for c in ("mean", "sd")}).hide(axis="index"))
display(pd.concat([category_counts(res, v) for v in (los_group, los30)])[["variable", "level", "count", "percent"]].style.format({"percent": PCT}).hide(axis="index"))
h = analysis([los], drop_missing=True).histogram(variable=los, bins=16).raw["histogram"][0]
days = pd.Series([c or 0 for c in h["counts"]], index=[str(i) if i < 15 else "15+" for i in range(16)])
ax = days.plot(kind="bar", color="#4C72B0", figsize=(8, 3.2), rot=0)
ax.set_xlabel("length of stay (days, top code 15)"); ax.set_ylabel("stays"); ax.set_title("Length of stay of 488,616 sampled discharges")
plt.tight_layout()
plt.show()

## Step 2. By reason for admission

One `describe()` per group for the ten most frequent first-listed diagnosis groups.

*Expected (median, mean):* pregnancy and childbirth 2 (2.68); liveborn infant 2 (2.79);
musculoskeletal 3 (4.03); digestive 4 (4.83); other circulatory 4 (5.09); poisoning and
complications of care 4 (5.74); hypertensive diseases 4 (5.78); neoplasms 4 (5.83); other mental
disorders 6 (6.85); sepsis 6 (7.26).

In [ ]:
top10 = ["2", "1", "26", "3", "6", "38", "28", "15", "20", "12"]
by_reason = by_level(los, group, levels=top10).sort_values("mean")
display(by_reason.style.format({"mean": "{:.2f}"}).hide(axis="index"))
short = by_reason[group.label].str.replace(r" \(.*\)", "", regex=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(short, by_reason["mean"], color="#4C72B0", label="mean")
ax.scatter(by_reason["median"], range(len(by_reason)), color="#C44E52", zorder=3, label="median")
ax.set_xlabel("days"); ax.set_title("Length of stay by first-listed diagnosis group (ten most frequent)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Step 3. By comorbidity

*Expected:* mean 3.50 days (median 2) with a Charlson index of 0, 5.09 (4) at 1-2, 5.92 (4) at
3-4, 6.37 (5) at 5 or more; one-way ANOVA F about 12,800, p < 0.001.

In [ ]:
display(by_level(los, charlson).style.format({"mean": "{:.2f}"}).hide(axis="index"))
anova = analysis([los, charlson], drop_missing=True).anova_oneway(group_by=charlson, outcome=los)
f, p, n, groups, tukey = anova_summary(anova)
print(f"One-way ANOVA of the length of stay across the four Charlson groups: F = {f:,.0f}, p = {fmt_p(p)}, n = {n:,}")

## Step 4. Childbirth

The deliveries (the mother's record): how they were delivered, by year and by the mother's age,
and how long they stayed.

*Expected:* 47,195 deliveries (9.7% of all discharges, the single commonest reason for a stay,
plus the 47,010 liveborn infants who get a stay of their own). A caesarean section is coded in
35.7% (16,862), a vaginal delivery in 61.5%, neither in 4.2%. The caesarean rate was 36.1%, 36.0%
and 35.1% in 2020, 2021 and 2022 (p = 0.11, no change), 13.0% in mothers under 18 (746
deliveries), 36.2% at 18 to 44, 66.1% at 45 to 64 (186). A caesarean stay lasts 3.3 days on
average against 2.26 (medians 3 and 2; t about 68). One US birth in three is a caesarean, a rate
twice the level the WHO regards as justified by medical need, stable across these years and rising
steeply with the mother's age.

In [ ]:
deliveries = F(delivery) == "1"
res = analysis([cesarean, vaginal, los], deliveries).describe()
k = category_counts(res, cesarean)
print(f"{int(k['count'].sum()):,} deliveries; caesarean section coded in {level_percent(res, cesarean):.1f}%, vaginal delivery in {level_percent(res, vaginal):.1f}%")
display(numeric_summary(res, [los]).style.format({c: "{:.2f}" for c in ("mean", "sd")}).hide(axis="index"))

by_year = crosstab(year, cesarean, base_filter=deliveries, event_label="caesarean")
print("Caesarean rate by year: " + ", ".join(f"{r[year.label]} {r['% caesarean']:.1f}%" for _, r in by_year.iterrows()) + f"; chi2 = {by_year.attrs['chi2']:.1f}, p = {fmt_p(by_year.attrs['p'])}")
show_crosstab(crosstab(age_group, cesarean, base_filter=deliveries, levels=["2", "3", "4"], event_label="caesarean"), "Caesarean rate by the mother's age group")
t = analysis([los, cesarean], deliveries, drop_missing=True).t_test(variable=los, group_by=cesarean, group_a="1", group_b="0")
s = ttest_summary(t)
print(f"Length of stay, caesarean minus vaginal or unrecorded: {s['mean_diff']:.2f} days, t = {s['t_stat']:.1f}, p = {fmt_p(s['p'])}")

## Step 5. Procedures, and the drugs of the pandemic

*Expected:* 63.7% of stays have at least one procedure; 36.3% none, 23.4% one, 15.7% two, 9.8%
three, 5.9% four, 8.8% five or more. Transfusion in 4.6%, imaging in 9.0%. A new-technology
procedure in 1.4% of stays in 2020, 3.4% in 2021 and 2.7% in 2022, and in 36.5% of stays with
COVID-19 against 0.7% of the others (22.8%, 49.2% and 33.0% of COVID-19 stays by year). Mean stay
3.8 days with no procedure, 4.1 with one, 4.5 with two, 5.3 with three or four, 9.0 with five or
more. "New technology" is the ICD-10-PCS section where the COVID-19 drugs were coded; its curve
across the years is the curve of remdesivir.

In [ ]:
res = analysis([pr_any, n_pr, transfusion, imaging, newtech]).describe()
print(f"Any procedure {level_percent(res, pr_any):.1f}%, transfusion {level_percent(res, transfusion):.1f}%, imaging {level_percent(res, imaging):.1f}%, new technology {level_percent(res, newtech):.1f}%")
display(numeric_summary(res, [n_pr]).style.format({c: "{:.2f}" for c in ("mean", "sd")}).hide(axis="index"))
by_year = rates_by(year, [newtech]).set_index("Survey year")
print("New technology by year: " + ", ".join(f"{y} {v:.1f}%" for y, v in by_year[newtech.label].items()))
show_crosstab(crosstab(covid, newtech, event_label="with a new-technology procedure"), "New-technology procedures, by COVID-19 diagnosis")
among_covid = rates_by(year, [newtech], base_filter=F(covid) == "1").set_index("Survey year")
print("Among COVID-19 stays: " + ", ".join(f"{y} {v:.1f}%" for y, v in among_covid[newtech.label].items()))
bands = {"0": F(n_pr) == 0, "1": F(n_pr) == 1, "2": F(n_pr) == 2, "3 to 4": F(n_pr).between(3, 4), "5 or more": F(n_pr) >= 5}
display(band_means(los, bands).style.format({"mean": "{:.2f}"}).hide(axis="index"))

## Step 6. A linear model of the stay

A *Linear Regression* of the length of stay on age, sex, Charlson group, COVID-19, sepsis and the
number of procedures, newborns excluded. A coefficient is a number of days added (or removed) for
the same values of the other predictors; categories are compared with their first code.

*Expected (437,123 discharges with a known stay, age, sex and condition flags):* intercept 2.20;
age +0.013 per year; women −0.43; Charlson 1-2
+0.98, 3-4 +1.54, 5 or more +1.83; COVID-19 +1.49; sepsis +2.29; each procedure +0.65 days.
R-squared 0.21, residual standard error 3.7 days. Sepsis adds more than two days, COVID-19 a day
and a half, comorbidity up to almost two, each procedure two thirds of a day; the model explains a
fifth of the variation: what keeps a patient in hospital is mostly in what the record does not
hold.

In [ ]:
cohort = F(newborn) == "2"
covariates = [age, sex, charlson, covid, sepsis, n_pr]
t0 = time.perf_counter()
model = analysis([los] + covariates, cohort, drop_missing=True).linear_regression(x=covariates, y=los)
raw = model.raw
print(f"{raw['n_obs']:,} discharges; R-squared {raw['r_squared']:.3f}, residual standard error {raw['rse']:.2f} days; "
      f"the platform answered in {time.perf_counter() - t0:.0f} s")
show_coefficients(linear_table(model, covariates), "Days of stay added by each factor, newborns excluded")

## Step 7. State the result

A model answer:

> Among 488,636 sampled US hospital discharges of 2020 to 2022, the median stay was 3 days
> (quartiles 2 to 6; 7% of stays reached the 15-day top code). Childbirth was the commonest reason
> for a stay, with a caesarean rate of 36% that did not change over the three years and rose with
> the mother's age. The stay lengthened with comorbidity (median 2 days at Charlson 0, 5 at 5 or
> more) and with the number of procedures; in a linear model, sepsis added 2.3 days, COVID-19 1.5,
> a Charlson index of 5 or more 1.8 and each procedure 0.65, for the same age and sex. The
> new-technology procedures of the pandemic, mostly remdesivir, were coded in half of the COVID-19
> stays of 2021. Unweighted sample statistics; the top code at 15 days flattens the long stays.

**Questions for discussion.** Mean 4.66 days and median 3: which one to plan beds with, and why
does the top code matter? The caesarean rate rises from 13% in mothers under 18 to 66% at 45 to
64: three reasons, clinical and not? The number of procedures tracks the stay as cause and as
consequence: should it be a predictor? Is 21% of the variation explained good or bad?

**Optional extension.** Discharge destination by length of stay: transfers to another facility are
the longest stays (mean 7.9 days), a sign of patients waiting for a bed elsewhere.

In [ ]:
display(by_level(los, outcome).sort_values("mean").style.format({"mean": "{:.2f}"}).hide(axis="index"))